# Projeto 2 - LIPAI
## Etapas
1) Redimensionar e normalizar as imagens
2) Criar nova pasta com data augmentation (2 tipos com e sem)
3) Carregar o modelo (2 tipos)
    - Colocar para classificar a quantidade de classes que temos
4) Treinamento (3 tipos) * (3 seeds)
5) Salvar os pesos

## Informações Importantes
### Arquiteturas
- EfficientNetV2-B0
- EfficientNetV2-B1

### Modos de Treinamento
- **From Scratch** - Treinar todos os pesso do modelo do 0
- **Pré-treinado com Backbone Congelado** - Utilizar pesos pré treinados, congelar backbone e treinar apenas camada final
- **Pré-treinado com Fine-tuning Completo** - Utilizar pesos pré-treinados e treinar todas as camadas do modelo, inclusindo backbone e classificador

### Data Augmentation
- Sem augmentation
- Com augmentation (RandomHorizontalFlip, RandomRotation)

### Seeds
- 42
- 123
- 2025

### Protocolos
- Imagens redimensionadas para 224x224
- Normalização padrão do ImageNet
    * mean = (0.485, 0.456, 0.406)
    * std  = (0.229, 0.224, 0.225)
- **Épocas:** 50 no máximo


### Resultados
- Para cada classe com augmentation precisa de 3 imagens exemplo

#### Para cada Execução deverá ter:
- Acurácia
- F1-score macro
- F1-score weighted
- Matriz de confusão
- Curvas de aprendizado:
    * loss de treino e validação por época
    * acurácia de treino e validação por época
- Para as métricas principais, deverão ser calculadas:
    * Média das 3 repetições (seeds)
    * Desvio padrão das 3 repetições (seeds)
- Métricas obrigatórias para média e desvio padrão
    * Acurácia
    * F1-score macro
    * F1-score weighted

### Planilha CSV
Contendo:
* repetition
* seed
* dataset
* model
* training_mode
* augmentation
* acc_test
* f1_macro_test
* f1_weighted_test
* num_params
* gflops
* best_epoch
* val_acc_best


## Gráficos
- Gráfico de barras do:
    * F1-score por arquitetura
    * modo de treinamento
    * uso de augmentation
- Tabela com média e desvio padrão das repetições
- Gráfico com número de parâmetros por arquitetura
- Gráfico com complexidade computacional em GFLOPs considerando entrada 224x224

### Links

https://huggingface.co/timm/tf_efficientnetv2_b0.in1k

https://huggingface.co/timm/tf_efficientnetv2_b1.in1k



# Código

## Importações

In [26]:
import timm
import pandas as pd
import torch
from torch import optim, nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import random
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from thop import profile
import copy
import json
import time

## Definições do projeto

- **datasets_folder:** pasta que contem os datasets
- **imgs1_folder:** pasta com as imagens do dataset 1
- **imgs2_folder:** pasta com as imagens do dataset 2
- **epochs:** número limite de épocas
- **batch_size:** tamanho do batch (número de imagens que vão entrar juntar para treinar)
- **patience:** limite de épocas que o modelo aguarda passar sem melhorar a acurácia para parar o treinamento (early stop)

In [2]:
definitions = {
    "datasets_folder": "../datasets/",
    "imgs1_folder": "../datasets/Original ROI images/",
    "imgs2_folder": "../datasets/images/",
    "epochs": 50,
    "batch_size": 32,
}


### Arquiteturas escolhidos

In [3]:
models = [
    'tf_efficientnetv2_b0.in1k',
    'tf_efficientnetv2_b1.in1k'
]

### Modos de treinamento
- **scratch:** Treina o modelo do 0
- **feature_extraction:** Importa o modelo pré treinado e ajusta apenas a última camada
- **fine_tuning:** Importa o modelo pré treinado e ajusta todas as camadas dele

In [4]:
training_modes = [
    "scratch",
    "feature_extraction",
    "fine_tuning"
]

### Aumento de Dados (data augmentation)
- **True** - Ligado
- **False** - Desligado

In [5]:
augmentation_modes = [
    True,
    False
]

### Sementes Utilizadas

In [6]:
seeds = [
    42,
    123,
    2025
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Importação dos Datasets

#### Dataset 1

In [7]:
df_epithelium = pd.read_csv(definitions["datasets_folder"] + "manifest_split_oralepitheliumdb.csv")
df_epithelium

,path,task,sets,label_number
0,healthy/healthy-11-roi3.tif,healthy,train,0
1,healthy/healthy-03-roi1.tif,healthy,train,0
2,healthy/healthy-20-roi1.tif,healthy,train,0
3,healthy/healthy-04-roi2.tif,healthy,train,0
4,healthy/healthy-13-roi4.tif,healthy,train,0
...,...,...,...,...
223,severe/severe-05-roi2.tif,severe,test,1
224,severe/severe-18-roi3.tif,severe,test,1
225,severe/severe-13-roi3.tif,severe,test,1
226,severe/severe-23-roi1.tif,severe,test,1


#### Dataset 2

In [8]:
df_cancer = pd.read_csv(definitions["datasets_folder"] + "manifest_split_multiclass_NDB-UFES.csv")
df_cancer

,path,task,sets,label_number
0,0005.png,OSCC,train,2
1,0006.png,Leukoplakia with dysplasia,train,0
2,0009.png,Leukoplakia with dysplasia,train,0
3,0012.png,OSCC,train,2
4,0013.png,OSCC,train,2
...,...,...,...,...
232,0193.png,OSCC,test,2
233,0200.png,Leukoplakia without dysplasia,test,1
234,0218.png,Leukoplakia with dysplasia,test,0
235,0224.png,OSCC,test,2


## Redimensionar e Normalizar

In [9]:
# Transformações para redimensionar e normalizar as imagens
# Além de dar 50% de probabilidade de inverter horizontalmente
# e pode rotacionar ou 0°/90°/180°/270° com probabilidades iguais
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomChoice([
        transforms.RandomRotation(degrees=(0, 0)),
        transforms.RandomRotation(degrees=(90, 90)),
        transforms.RandomRotation(degrees=(180, 180)),
        transforms.RandomRotation(degrees=(270, 270)),
    ]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Transformação para validação
# Apenas redimensiona e normaliza as imagens
transform_validation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Criação do Dataset

In [10]:
class OralCancerDataset(Dataset):
    def __init__(self, df, images_folder, transform=None):
        self.df = df
        self.images_folder = images_folder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_folder, row["path"])
        image = Image.open(img_path).convert("RGB")
        label = row["label_number"]

        if self.transform:
            image = self.transform(image)

        return image, label

## Separação dos dados em treino, validação e teste

Separando manualmente os dataframes de treino, validação e teste usando `pandas`

#### Dataset 1

In [11]:
df_epithelium_train = df_epithelium[df_epithelium["sets"] == "train"].reset_index(drop=True)
df_epithelium_val   = df_epithelium[df_epithelium["sets"] == "val"].reset_index(drop=True)
df_epithelium_test  = df_epithelium[df_epithelium["sets"] == "test"].reset_index(drop=True)

#### Dataset 2

In [12]:
df_cancer_train = df_cancer[df_cancer["sets"] == "train"].reset_index(drop=True)
df_cancer_val   = df_cancer[df_cancer["sets"] == "val"].reset_index(drop=True)
df_cancer_test  = df_cancer[df_cancer["sets"] == "test"].reset_index(drop=True)

## Criação dos diferentes tipos de treinamento

In [13]:
def create_model(architecture, mode, num_classes=3):
    model = None
    if mode == "scratch":
        model = timm.create_model(architecture, pretrained=False, num_classes=num_classes)

    elif mode == "feature_extraction":
        model = timm.create_model(architecture, pretrained=True, num_classes=num_classes)
        for param in model.parameters():
            param.requires_grad = False
        for param in model.get_classifier().parameters():
            param.requires_grad = True

    elif mode == "fine_tuning":
        model = timm.create_model(architecture, pretrained=True, num_classes=num_classes)

    return model

In [14]:
def get_transform_train(augmentation=True):
    if augmentation:
        return transform_train
    else:
        return transform_validation

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {device}")

Usando: cpu


## Definições pré-treinamento

In [16]:
datasets = [
    # Dataset 1
    {
        "name": "OralEpitheliumDB",
        "train": df_epithelium_train,
        "val": df_epithelium_val,
        "test": df_epithelium_test,
        "folder": definitions["imgs1_folder"],
        "num_classes": 2
    },
    # Dataset 2
    {
        "name": "NDB-UFES",
        "train": df_cancer_train,
        "val": df_cancer_val,
        "test": df_cancer_test,
        "folder": definitions["imgs2_folder"],
        "num_classes": 3
    },

]

## Treinamento

In [17]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

          # Train Loss                Train Accuracy
    return total_loss / len(loader), correct / total

In [18]:
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [19]:
def evaluate_test(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return {
        "acc_test":          accuracy_score(all_labels, all_preds),
        "f1_macro_test":     f1_score(all_labels, all_preds, average="macro"),
        "f1_weighted_test":  f1_score(all_labels, all_preds, average="weighted"),
        "confusion_matrix":  confusion_matrix(all_labels, all_preds).tolist()
    }

In [20]:
def get_model_complexity(model, loader):
    num_params = sum(p.numel() for p in model.parameters())
    sample, _ = next(iter(loader))
    dummy = sample[0].unsqueeze(0).to(device)
    flops, _ = profile(model, inputs=(dummy,), verbose=False)
    return num_params, flops / 1e9

In [21]:
def save_checkpoint(dataset_name, weights, model_name, mode, aug, seed):
    os.makedirs("../checkpoints", exist_ok=True)
    path = f"../checkpoints/{dataset_name}_{model_name}_{mode}_aug{aug}_seed{seed}.pth"
    torch.save(weights, path)

In [22]:
def save_execution(dataset_name, model_name, mode, aug, seed, history, test_metrics):
    name = f"{dataset_name}_{model_name}_{mode}_aug{aug}_seed{seed}"
    os.makedirs("../histories", exist_ok=True)
    os.makedirs("../confusion_matrices", exist_ok=True)

    with open(f"../histories/{name}.json", "w") as file:
        json.dump(history, file)

    with open(f"../confusion_matrices/{name}.json", "w") as file:
        json.dump(test_metrics["confusion_matrix"], file)

In [23]:
def training(model, loader_train, loader_val, loader_test, criterion, optimizer, epochs, model_name, mode, aug, seed, dataset_name):

    best_val_acc, best_epoch = 0.0, 0
    best_weights = copy.deepcopy(model.state_dict())
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, loader_train, criterion, optimizer)
        val_loss, val_acc     = validate(model, loader_val, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} | train_acc: {train_acc:.4f} | val_acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            best_weights = copy.deepcopy(model.state_dict())




    save_checkpoint(dataset_name, best_weights, model_name, mode, aug, seed)
    model.load_state_dict(best_weights)

    test_metrics = evaluate_test(model, loader_test)
    num_params, gflops = get_model_complexity(model, loader_train)

    save_execution(dataset_name, model_name, mode, aug, seed, history, test_metrics)

    return {
        "seed": seed,
        "dataset": dataset_name,
        "model": model_name,
        "training_mode": mode,
        "augmentation": aug,
        "best_epoch": best_epoch,
        "val_acc_best": best_val_acc,
        "num_params": num_params,
        "gflops": gflops,
        **test_metrics
    }

In [24]:
tempo_inicio = time.time()

results = []
for ds in datasets:
    for seed in seeds:
        for model_name in models:
            for mode in training_modes:
                for aug in augmentation_modes:
                    print(f"\n=== Dataset: {ds['name']} | Seed: {seed} | Model: {model_name} | Mode: {mode} | Aug: {aug} ===")

                    set_seed(seed)

                    # Loaders
                    transform = get_transform_train(aug)
                    dataset_train = OralCancerDataset(ds["train"], ds["folder"], transform=transform)
                    dataset_val   = OralCancerDataset(ds["val"], ds["folder"], transform=transform_validation)
                    dataset_test  = OralCancerDataset(ds["test"], ds["folder"], transform=transform_validation)

                    loader_train = DataLoader(dataset_train, batch_size=definitions["batch_size"], shuffle=True)
                    loader_val   = DataLoader(dataset_val,   batch_size=definitions["batch_size"], shuffle=False)
                    loader_test  = DataLoader(dataset_test,  batch_size=definitions["batch_size"], shuffle=False)

                    # Model
                    model = create_model(model_name, mode, num_classes=ds["num_classes"])
                    model = model.to(device)

                    # Optimizer and loss
                    criterion = nn.CrossEntropyLoss()
                    optimizer = optim.Adam(model.parameters(), lr=1e-4)

                    # Train
                    result = training(model, loader_train, loader_val, loader_test, criterion, optimizer, epochs=definitions["epochs"], model_name=model_name, mode=mode, aug=aug, seed=seed, dataset_name=ds["name"])

                    results.append(result)

tempo_fim = time.time()

# Save in Csv
df_results = pd.DataFrame(results)
df_results.to_csv("../results.csv", index=False)


=== Dataset: OralEpitheliumDB | Seed: 42 | Model: tf_efficientnetv2_b0.in1k | Mode: scratch | Aug: True ===


KeyboardInterrupt: 

In [ ]:
tempo_total = tempo_fim - tempo_inicio

print(f"O código demorou {tempo_total:.2f} segundos para executar.")